# Lab 11: Chain-of-Thought

## Making the model show its work

In this notebook you build three helpers. `build_direct_prompt` asks for a short answer right away. `build_cot_prompt` asks the model to think step by step, then end with a line `"Answer: <answer>"`. `extract_final_answer` pulls that last line back out. The Reading tab explains what a reasoning trace is. It also explains why it helps to compare a direct answer with a step-by-step one.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

### Goal
Sometimes a model answers a hard question better if you ask it to reason out loud first. This is called *chain-of-thought* prompting (CoT for short). You write three small helpers. One makes a prompt that asks for a quick answer. One makes a prompt that asks for step-by-step thinking, ending with a marked final answer. One picks the final answer out of a long reply. You do not call any API here. You only build and read plain Python values.

A *message* is one chat turn, written as a dict like `{"role": "user", "content": "Hello"}`. A prompt here is a list that holds one such message.

### 1. `build_direct_prompt(question)`
Code shape (replace each `...`):
```python
def build_direct_prompt(question):
    text = ...        # the question plus a "short answer" instruction
    return [{"role": "user", "content": ...}]
```
You get `question`, a string like `"What is 12 * 13?"`. You give back a list with one user message, like `[{"role": "user", "content": "What is 12 * 13? Answer in as few words as possible."}]` (your words may differ).

Write the code that does these steps:

1. Set `text` to the question, a space, then a short-answer request such as `"Answer in as few words as possible."`.
2. Return a list with one dict: role `"user"` and content `text`.

**Check:** `len(build_direct_prompt("Hi"))` is `1`, the item's `"role"` is `"user"`, and its `"content"` contains `"Hi"`.

### 2. `build_cot_prompt(question)`
Code shape:
```python
def build_cot_prompt(question):
    text = ...        # the question, then "step by step", then the Answer: line request
    return [{"role": "user", "content": ...}]
```
You get `question`, a string like `"What is 12 * 13?"`. You give back a list with one user message. Its `Answer:` line is the marker that lets `extract_final_answer` find the result later.

Write the code that does these steps:

1. Start `text` with `question`.
2. Add an instruction to think `step by step`. Use exactly that phrase, because the checks look for it.
3. Add a request to end with a last line like `Answer: <final answer>`.
4. Return a list with one dict: role `"user"` and content `text`.

**Check:** `"step by step" in build_cot_prompt("What is 12 * 13?")[0]["content"].lower()` is `True`, and the content also has `"Answer:"`.

### 3. `extract_final_answer(reply_text, marker="Answer:")`
Code shape:
```python
def extract_final_answer(reply_text, marker="Answer:"):
    if ... not in reply_text:
        return ...
    after = reply_text.rsplit(..., 1)[...]
    return ...
```
You get `reply_text`, the model's whole reply (maybe many lines), and `marker`, which is `"Answer:"` unless the caller gives another one. For `"Let's see... 12*13=156.\nAnswer: 156"` you give back `"156"`.

Write the code that does these steps:

1. If `marker` is not in `reply_text`, return `reply_text.strip()`.
2. Otherwise, split with `reply_text.rsplit(marker, 1)` and take the piece after the split (index `1`). It splits at the last marker, so `"Answer: 1\nAnswer: 2"` gives `"2"`.
3. Return that piece with `.strip()`.

The match is exact, so `"answer:"` in lowercase does not match.

**Check:** `extract_final_answer("so... Answer: yes")` is `"yes"`, `extract_final_answer("just text")` is `"just text"`, and `extract_final_answer("Reasoning\nFinal: 7", "Final:")` is `"7"`.

### What the checks below test
The cell below the stubs runs these checks:
- the chain-of-thought prompt's content has `step by step` in it (ignoring capital letters);
- taking the answer from `"Let's see... 12*13=156.\nAnswer: 156"` gives `"156"`;
- a reply with no marker comes back unchanged (`"no marker here"`).

If every check passes, you see `All checks passed.` printed. If a check fails, the message tells you what it expected and what your function gave back. If you see `NotImplementedError`, replace the `TODO` and the `raise` line in that function with your code.

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    """Build a prompt that asks the question directly and wants only a short answer.

    You get:
        question: the question text, like "What is 12 * 13?".

    You give back:
        A list with ONE message dict, like
        [{"role": "user", "content": "What is 12 * 13? Answer in as few words as possible."}]

    Steps:
        1. Write an instruction that asks for the final answer only, in as
           few words as possible.
        2. Join the question and that instruction into one string.
        3. Give back a list with one item: {"role": "user", "content": <that string>}.
    """
    raise NotImplementedError("build_direct_prompt() is not implemented yet - replace the TODO with your code")


def build_cot_prompt(question: str) -> list[dict]:
    """Build a prompt that asks the model to think step by step first.

    You get:
        question: the question text, like "What is 12 * 13?".

    You give back:
        A list with ONE user message dict. Its content has the question, the
        phrase "step by step", and a request to end the reply with a last
        line that looks like 'Answer: <final answer>'.

    Steps:
        1. Start with the question text.
        2. Add an instruction to think step by step (use that exact phrase).
        3. Add an instruction to end with a last line 'Answer: <your answer>'.
        4. Give back a list with one item: {"role": "user", "content": <the text>}.
    """
    raise NotImplementedError("build_cot_prompt() is not implemented yet - replace the TODO with your code")


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    """Pick the final answer out of a model reply.

    You get:
        reply_text: the whole reply, like "Let's see... 12*13=156." followed
            by a new line and "Answer: 156".
        marker: the label that comes before the answer (default "Answer:").

    You give back:
        Everything after the LAST `marker`, with spaces and new lines removed
        from both ends, like "156". If `marker` is not in the reply, you give
        back the whole reply with the spaces removed, like "no marker here".

    Steps:
        1. Check if `marker` is in `reply_text` (use `in`).
        2. If it is not, give back `reply_text.strip()`.
        3. If it is, split at the last marker (rsplit with a limit of 1)
           and take the piece after it.
        4. Remove the spaces from that piece and give it back.
    """
    raise NotImplementedError("extract_final_answer() is not implemented yet - replace the TODO with your code")


assert "step by step" in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), 'expected ' + repr("step by step") + ' to be in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), but it is ' + repr(build_cot_prompt("What is 12 * 13?")[0]["content"].lower())
assert extract_final_answer("Let's see... 12*13=156.\nAnswer: 156") == "156", 'extract_final_answer("Let\'s see... 12*13=156.\\nAnswer: 156") should be ' + repr("156") + ", got " + repr(extract_final_answer("Let's see... 12*13=156.\nAnswer: 156"))
assert extract_final_answer("no marker here") == "no marker here", 'extract_final_answer("no marker here") should be ' + repr("no marker here") + ", got " + repr(extract_final_answer("no marker here"))

print("All checks passed.")

## Solution

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    return [{"role": "user", "content": f"{question} Answer in as few words as possible."}]


def build_cot_prompt(question: str) -> list[dict]:
    return [{
        "role": "user",
        "content": f"{question}\n\nThink step by step, then end your reply with a final "
                   "line in exactly this form: 'Answer: <your answer>'",
    }]


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    if marker not in reply_text:
        return reply_text.strip()
    return reply_text.rsplit(marker, 1)[1].strip()


print(build_cot_prompt("What is 12 * 13?"))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

You compare a direct answer with a chain-of-thought answer on a few small reasoning questions. Small, fast models often get these wrong when they rush to answer in one shot.

In [ ]:
QUESTIONS = [
    ("If a train travels 60 miles in 45 minutes, what is its speed in miles per hour?", "80"),
    ("A shirt costs $40 after a 20% discount. What was the original price?", "50"),
    ("Sam has 3 times as many apples as Jo. Together they have 24 apples. How many does Sam have?", "18"),
]

for question, expected in QUESTIONS:
    direct_reply = call_groq(build_direct_prompt(question))
    cot_reply = call_groq(build_cot_prompt(question))
    cot_answer = extract_final_answer(cot_reply)

    print(f"Q: {question}")
    print(f"  expected:            {expected}")
    print(f"  direct answer:       {direct_reply.strip()[:60]}")
    print(f"  chain-of-thought:    {cot_answer}")
    print()